In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, 
    confusion_matrix, 
    classification_report, 
    roc_curve,auc
)

def main():
    try:
        df = pd.read_csv('heart.csv')
        print("Dataset loaded successfully!")
    except FileNotFoundError:
        print("Error: 'heart.csv' not found. Please download it from Kaggle and place it in the same folder.")
        return

    df = df.dropna() 
    df = df.drop_duplicates()
    
    print(f"Dataset shape after cleaning: {df.shape}")

    plt.figure(figsize=(16, 5))

    plt.subplot(1, 2, 1)
    sns.countplot(x='target', data=df, palette='Set2')
    plt.title('Heart Disease Distribution (0 = No Disease, 1 = Disease)')
    plt.xlabel('Target')
    plt.ylabel('Count')

   
    plt.subplot(1, 2, 2)
    corr_matrix = df.corr()
    sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', linewidths=0.5)
    plt.title('Feature Correlation Heatmap')
    
    plt.tight_layout()
    plt.show()


    X = df.drop('target', axis=1)
    y = df['target']

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    
  
    dt_model = DecisionTreeClassifier(max_depth=5, random_state=42)
    dt_model.fit(X_train, y_train) 
    dt_preds = dt_model.predict(X_test)
    dt_probs = dt_model.predict_proba(X_test)[:, 1]

    lr_model = LogisticRegression(max_iter=1000, random_state=42)
    lr_model.fit(X_train_scaled, y_train) 
    lr_preds = lr_model.predict(X_test_scaled)
    lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]


    print("\n--- Decision Tree Performance ---")
    print(f"Accuracy: {accuracy_score(y_test, dt_preds):.4f}")
    print("\nClassification Report:\n", classification_report(y_test, dt_preds))

    print("\n--- Logistic Regression Performance ---")
    print(f"Accuracy: {accuracy_score(y_test, lr_preds):.4f}")
    print("\nClassification Report:\n", classification_report(y_test, lr_preds))

    plt.figure(figsize=(16, 5))

    plt.subplot(1, 2, 1)
    cm = confusion_matrix(y_test, lr_preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title('Confusion Matrix (Logistic Regression)')
    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')

    plt.subplot(1, 2, 2)
    fpr_dt, tpr_dt, _ = roc_curve(y_test, dt_probs)
    roc_auc_dt = auc(fpr_dt, tpr_dt)
    
    fpr_lr, tpr_lr, _ = roc_curve(y_test, lr_probs)
    roc_auc_lr = auc(fpr_lr, tpr_lr)

    plt.plot(fpr_dt, tpr_dt, color='green', lw=2, label=f'Decision Tree (AUC = {roc_auc_dt:.2f})')
    plt.plot(fpr_lr, tpr_lr, color='blue', lw=2, label=f'Logistic Reg (AUC = {roc_auc_lr:.2f})')
    plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC) Curve')
    plt.legend(loc="lower right")
    
    plt.tight_layout()
    plt.show()

    feature_importances = pd.Series(
        dt_model.feature_importances_, 
        index=X.columns
    ).sort_values(ascending=False)

    plt.figure(figsize=(10, 6))
    sns.barplot(x=feature_importances.values, y=feature_importances.index, palette='viridis')
    plt.title('Feature Importance (Decision Tree)')
    plt.xlabel('Importance Score')
    plt.ylabel('Features')
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()